In [1]:
import pandas as pd
import numpy as np
import matplotlib as plot
import seaborn

In [2]:
df = pd.read_csv("variants.csv", sep='\t')
print(df.head())

                        Type                                            Name  \
0                   Deletion  NM_014855.3(AP5Z1):c.1413_1426del (p.Leu473fs)   
1                   Deletion  NM_014855.3(AP5Z1):c.1413_1426del (p.Leu473fs)   
2  single nucleotide variant     NM_017547.4(FOXRED1):c.694C>T (p.Gln232Ter)   
3  single nucleotide variant     NM_017547.4(FOXRED1):c.694C>T (p.Gln232Ter)   
4  single nucleotide variant    NM_017547.4(FOXRED1):c.1289A>G (p.Asn430Ser)   

  GeneSymbol ClinicalSignificance  \
0      AP5Z1           Pathogenic   
1      AP5Z1           Pathogenic   
2    FOXRED1           Pathogenic   
3    FOXRED1           Pathogenic   
4    FOXRED1    Likely pathogenic   

                                       PhenotypeList    Origin Assembly  \
0                   Hereditary spastic paraplegia 48  germline   GRCh37   
1                   Hereditary spastic paraplegia 48  germline   GRCh38   
2  Mitochondrial complex I deficiency, nuclear ty...  germline   GRCh37

In [3]:
target_map = {
    'Pathogenic': 1,
    'Likely pathogenic': 1,
    'Benign': 0,
    'Likely benign': 0
}

In [4]:
df['target'] = df['ClinicalSignificance'].map(target_map)
# Bilinmeyen veya VUS olanları temizleyelim (Model net öğrensin)
df = df.dropna(subset=['target']).copy()


In [5]:
df = pd.get_dummies(df, columns=['Type', 'Origin', 'Assembly'], prefix=['type', 'origin', 'asm'])
# Gen isimlerine 1, 2, 3... gibi numaralar verir
df['GeneSymbol_encoded'] = df['GeneSymbol'].astype('category').cat.codes

In [6]:

# VCF kolonlarındaki harf uzunluklarını ölçüyoruz
df['ref_len'] = df['ReferenceAlleleVCF'].str.len()
df['alt_len'] = df['AlternateAlleleVCF'].str.len()
df['diff_len'] = df['alt_len'] - df['ref_len'] # Negatifse silinme (deletion), pozitifse ekleme (insertion)

In [7]:
# Artık sayısal karşılığı olan veya gereksiz metin kolonlarını atıyoruz
cols_to_drop = [
    'Name', 'GeneSymbol', 'ClinicalSignificance', 'PhenotypeList',
    'ReferenceAllele', 'AlternateAllele', 'ReferenceAlleleVCF', 'AlternateAlleleVCF'
]
df_final = df.drop(columns=cols_to_drop)

In [8]:
# Sonuca bakalım
print(df_final.head())

numeric_cols = df.select_dtypes(include=['bool', 'number']).columns

# 2. Sadece bu kolonları int'e çevir
df[numeric_cols] = df[numeric_cols].astype(int)

   target  type_Complex  type_Deletion  type_Duplication  type_Indel  \
0       1         False           True             False       False   
1       1         False           True             False       False   
2       1         False          False             False       False   
3       1         False          False             False       False   
4       1         False          False             False       False   

   type_Insertion  type_Inversion  type_Microsatellite  type_Translocation  \
0           False           False                False               False   
1           False           False                False               False   
2           False           False                False               False   
3           False           False                False               False   
4           False           False                False               False   

   type_Variation  ...  origin_uniparental  origin_unknown  asm_GRCh37  \
0           False  ...  

In [9]:
pathogenic_group = df[df['target'] == 1]
benign_group = df[df['target'] == 0]

print(f"Total number of Pathogeneic: {len(pathogenic_group)}")
print(f"Total number of Benign: {len(benign_group)}")

# 2. Her gruptan rastgele 10.000'er örnek seç (Sampling)
# replace=False diyerek aynı veriyi iki kez seçmiyoruz
n_samples = 10000

Total number of Pathogeneic: 626901
Total number of Benign: 2541391


In [10]:
if len(pathogenic_group) >= n_samples and len(benign_group) >= n_samples:
    path_sampled = pathogenic_group.sample(n=n_samples, random_state=42)
    benign_sampled = benign_group.sample(n=n_samples, random_state=42)

    # 3. İki grubu birleştir
    df_balanced = pd.concat([path_sampled, benign_sampled])

    # 4. Veriyi karıştır (Shuffle) - Hepsi alt alta gelmesin
    df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

    print("\n20k'lık Dengeli Veri Seti Oluşturuldu!")
    print(df_balanced['target'].value_counts())

    # 5. Kaydet
    df_balanced.to_csv("variants_new_20k.csv", index=False)
else:
    print("HATA: Gruplardan birinde 10.000 veri bulunmuyor. Lütfen n_samples değerini düşürün.")


20k'lık Dengeli Veri Seti Oluşturuldu!
target
0    10000
1    10000
Name: count, dtype: int64
